# A PPO update in NumPy

A news app shows each reader one of four headline styles, and its policy has one row of
logits per reader group. An older copy of the policy, `theta_old`, collected a batch of 90
readers, and the advantages are already computed. You will write PPO's clipped objective,
its exact gradient with respect to the logits, and an update that makes many passes over the
same batch, then compare how far the policy moves with the clip and without it.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np


def log_softmax_rows(theta):
    """Log-probabilities for every row of logits, computed stably."""
    shifted = theta - np.max(theta, axis=-1, keepdims=True)
    return shifted - np.log(np.sum(np.exp(shifted), axis=-1, keepdims=True))


def central_differences(f, P, h=1e-6):
    """Central differences of f() in every entry of the array P, which is nudged in place and restored."""
    out = np.zeros_like(P)
    for i in np.ndindex(P.shape):
        keep = P[i]
        P[i] = keep + h
        f_up = f()
        P[i] = keep - h
        f_down = f()
        P[i] = keep
        out[i] = (f_up - f_down) / (2 * h)
    return out


# Three reader groups, four headline styles. theta_old is the policy that collected the batch,
# and click_rate is the chance that a reader of each group clicks on each style.
rng = np.random.default_rng(12)
N_GROUPS, N_STYLES, N = 3, 4, 90
theta_old = rng.normal(0.0, 0.5, size=(N_GROUPS, N_STYLES))
click_rate = np.array([[0.30, 0.10, 0.20, 0.05], [0.05, 0.25, 0.10, 0.15], [0.10, 0.10, 0.05, 0.35]])
pi_old = np.exp(log_softmax_rows(theta_old))
states = rng.integers(0, N_GROUPS, size=N)
actions = np.array([rng.choice(N_STYLES, p=pi_old[s]) for s in states])
clicks = (rng.random(N) < click_rate[states, actions]).astype(float)
critic = (pi_old * click_rate).sum(axis=1)          # each group's click rate under theta_old

batch = {
    "states": states,                                          # the reader's group
    "actions": actions,                                        # the style shown
    "logp_old": log_softmax_rows(theta_old)[states, actions],  # log pi_old(a | s) when it was shown
    "adv": clicks - critic[states],                            # the advantage of each sample
}
print(f"{N} readers and {int(clicks.sum())} clicks, samples per group {np.bincount(states)}")

## Part 1 (coding task): The clipped objective

Write `ppo_objective(theta, batch, eps)`. For each sample the ratio is
$r = \pi_\theta(a \mid s) / \pi_{\theta_\text{old}}(a \mid s)$, computed from the stored
`logp_old`, and the function returns the mean over the batch of

$$
\min\big(r A,\ \operatorname{clip}(r, 1 - \epsilon, 1 + \epsilon)\, A\big) .
$$

In [ ]:
def ppo_objective(theta, batch, eps):
    """Mean over the batch of min(r * A, clip(r, 1 - eps, 1 + eps) * A)."""
    # TODO
    raise NotImplementedError

In [ ]:
tiny = {
    "states": np.zeros(5, dtype=int),
    "actions": np.array([1, 0, 0, 1, 1]),
    "logp_old": np.log([0.6, 0.5, 0.2, 0.9, 0.5]),
    "adv": np.array([2.0, 1.0, -1.0, -2.0, 0.5]),
}
theta_tiny = np.log(np.array([[1.0, 3.0]]))              # the new policy is (0.25, 0.75)
value = ppo_objective(theta_tiny, tiny, 0.2)
mistakes = {
    0.18666667: "every ratio outside [0.8, 1.2] got the clipped term: take the smaller of r*A and clip(r)*A",
    0.23666667: "this takes the larger of r*A and clip(r)*A",
    0.16666667: "nothing was clipped: clip the ratio to [1 - eps, 1 + eps] in the second term",
    0.58333333: "this is the sum over the batch: return the mean",
    -0.01333333: "the ratio is upside down: divide the new probability by the old one",
}
for wrong, message in mistakes.items():
    assert not np.isclose(value, wrong, atol=1e-6), message
assert np.isclose(value, 7 / 60), f"objective {value:.6f} on a five-sample batch, expected {7 / 60:.6f}"
assert np.isclose(ppo_objective(theta_old, batch, 0.2), batch["adv"].mean()), "at theta_old every ratio is 1, so the objective is the mean advantage"
print("Part 1 passed")

## Part 2 (coding task): Its gradient

Write `ppo_gradient(theta, batch, eps)`, the exact gradient of `ppo_objective` with respect
to `theta`, as an array shaped like `theta`. Several samples share each row of `theta`. The
check compares your gradient with a hand-worked one on the five-sample batch, then with
central differences of your `ppo_objective` at a `theta` where some ratios have left
$[1 - \epsilon, 1 + \epsilon]$.

In [ ]:
def ppo_gradient(theta, batch, eps):
    """Gradient of ppo_objective(theta, batch, eps) with respect to theta."""
    # TODO
    raise NotImplementedError

In [ ]:
g = np.asarray(ppo_gradient(theta_tiny, tiny, 0.2), dtype=float)
assert g.shape == theta_tiny.shape, f"the gradient has shape {g.shape}, expected {theta_tiny.shape}"
mistakes = {
    0.08333333: "only the sample inside [0.8, 1.2] passed a gradient: a sample outside it keeps its gradient when the minimum picks r*A",
    -0.07916667: "these are the gradients of the larger term: the objective takes the smaller one",
    0.1: "each sample's term is weighted by A alone: the gradient of r*A is r*A times the gradient of log pi",
    0.0: "nothing reached the row: with fancy indexing, grad[rows] += values adds once per distinct row, so use np.add.at",
}
for wrong, message in mistakes.items():
    assert not np.allclose(g, [[wrong, -wrong]], atol=1e-6), message
assert np.allclose(g, [[-7 / 240, 7 / 240]]), f"gradient {np.round(g, 6)} on the five-sample batch, expected {np.round([[-7 / 240, 7 / 240]], 6)}"

theta_m = theta_old + np.random.default_rng(3).normal(0.0, 0.4, size=theta_old.shape)
r_m = np.exp(log_softmax_rows(theta_m)[batch["states"], batch["actions"]] - batch["logp_old"])
g = np.asarray(ppo_gradient(theta_m, batch, 0.2), dtype=float)
num = central_differences(lambda: ppo_objective(theta_m, batch, 0.2), theta_m)
assert np.allclose(g, num, atol=1e-7), f"with {int(np.sum((r_m < 0.8) | (r_m > 1.2)))} of {N} ratios outside [0.8, 1.2], the gradient is\n{np.round(g, 5)}\nbut central differences give\n{np.round(num, 5)}"
print("Part 2 passed")

## Part 3 (coding task): Several passes over one batch

Write `ppo_update(theta_old, batch, eps, lr, epochs)`. Starting from a copy of `theta_old`, it
takes `epochs` steps of gradient ascent on `ppo_objective` over the whole batch, with step
size `lr`, and returns the new logits. The check runs 100 passes with the clip and 100
without it (a huge `eps` switches it off) and compares how far the ratios end up from 1.

In [ ]:
def ppo_update(theta_old, batch, eps, lr, epochs):
    """Run `epochs` full-batch gradient ascent steps on the clipped objective. Returns the new logits."""
    # TODO
    raise NotImplementedError

In [ ]:
theta_before = theta_old.copy()
one_pass = np.asarray(ppo_update(theta_old, batch, 0.2, 1.0, 1), dtype=float)
assert np.array_equal(theta_old, theta_before), "ppo_update changed theta_old: work on a copy"
step = ppo_gradient(theta_old, batch, 0.2)
assert not np.allclose(one_pass, theta_old - step), "one pass moved downhill: step uphill, theta += lr * gradient"
assert np.allclose(one_pass, theta_old + step), "one pass with lr = 1 should add ppo_gradient(theta_old, batch, eps) to theta_old"
theta_clip = np.asarray(ppo_update(theta_old, batch, 0.2, 1.0, 100), dtype=float)
theta_free = np.asarray(ppo_update(theta_old, batch, 1e9, 1.0, 100), dtype=float)


def ratio_range(theta):
    r = np.exp(log_softmax_rows(theta)[batch["states"], batch["actions"]] - batch["logp_old"])
    return r.min(), r.max()


lo_c, hi_c = ratio_range(theta_clip)
lo_f, hi_f = ratio_range(theta_free)
start, end = ppo_objective(theta_old, batch, 0.2), ppo_objective(theta_clip, batch, 0.2)
print(f"with the clip:    objective {start:.4f} -> {end:.4f}, ratios in [{lo_c:.2f}, {hi_c:.2f}]")
print(f"without the clip: ratios in [{lo_f:.2f}, {hi_f:.2f}]")
assert hi_f > 3, "without the clip, 100 passes should push some ratio past 3: is eps passed through to the gradient?"
assert 0.5 < lo_c and hi_c < 1.45, f"with the clip the ratios ended in [{lo_c:.2f}, {hi_c:.2f}]: recompute the gradient at the current theta on every pass"
assert end > start + 0.01, f"100 passes raised the clipped objective only from {start:.4f} to {end:.4f}"
print("Part 3 passed")

## Solution

### Part 1: The clipped objective

In [ ]:
def ppo_objective(theta, batch, eps):
    """Mean over the batch of min(r * A, clip(r, 1 - eps, 1 + eps) * A)."""
    logp = log_softmax_rows(np.asarray(theta, dtype=float))[batch["states"], batch["actions"]]
    ratio = np.exp(logp - batch["logp_old"])
    adv = batch["adv"]
    return float(np.mean(np.minimum(ratio * adv, np.clip(ratio, 1 - eps, 1 + eps) * adv)))

**Why it works.** Subtracting the stored log-probability and exponentiating gives the
ratio without dividing two small probabilities. On the five-sample batch the ratios are 1.25,
0.5, 1.25, 0.83 and 1.5, and the minimum keeps 2.4, 0.5, $-1.25$, $-1.67$ and 0.6, whose mean
is $7/60$. Samples 1 and 5 are capped by the clip because their advantage is positive and
their ratio is above 1.2. Samples 2 and 3 keep the unclipped term because it is the smaller
one: their ratio has moved the way their advantage does not favor. Sample 4 keeps its unclipped term too, but only because its ratio, 0.83, is
the band, where the clip changes nothing.

### Part 2: Its gradient

In [ ]:
def ppo_gradient(theta, batch, eps):
    """Gradient of ppo_objective(theta, batch, eps) with respect to theta."""
    theta = np.asarray(theta, dtype=float)
    s, a, adv = batch["states"], batch["actions"], batch["adv"]
    logp_all = log_softmax_rows(theta)
    ratio = np.exp(logp_all[s, a] - batch["logp_old"])
    flat = ((adv > 0) & (ratio > 1 + eps)) | ((adv < 0) & (ratio < 1 - eps))   # min picks the constant
    coef = np.where(flat, 0.0, ratio * adv) / len(adv)       # d(r A) = r A d(log pi)
    score = -np.exp(logp_all[s])
    score[np.arange(len(a)), a] += 1.0                       # one-hot(a) minus the policy
    grad = np.zeros_like(theta)
    np.add.at(grad, s, coef[:, None] * score)                # rows repeat, so add every sample
    return grad

**Why it works.** Each sample's term is either the constant
$\operatorname{clip}(r)A$, when the minimum picks it with $r$ outside the band, or $rA$.
Since $\nabla r = r\, \nabla \log \pi_\theta(a \mid s)$, the second has gradient
$rA\,(\text{one-hot}(a) - \pi_\theta(\cdot \mid s))$ in row $s$. The term is constant exactly
when $A > 0$ and $r > 1 + \epsilon$, or $A < 0$ and $r < 1 - \epsilon$: the ratio has gone past
the band on the side its advantage favors. On the five-sample batch that switches off samples
1 and 5, and samples 2, 3 and 4 add $0.375$, $-0.9375$ and $0.4167$ to the first logit, which
over the five samples averages to $-7/240$. `np.add.at` adds every sample to its group's row, which matters here
because all five share one row.

### Part 3: Several passes over one batch

In [ ]:
def ppo_update(theta_old, batch, eps, lr, epochs):
    """Run `epochs` full-batch gradient ascent steps on the clipped objective. Returns the new logits."""
    theta = np.array(theta_old, dtype=float)
    for _ in range(epochs):
        theta += lr * ppo_gradient(theta, batch, eps)
    return theta

**Why it works.** Every pass recomputes the ratios, so a sample drops out once its
ratio passes the band on the side its advantage favors, and the policy stops moving that
way. The ratios still end a little outside $[0.8, 1.2]$ because samples share rows: lowering
one style in a group raises the others, including styles whose samples have already dropped
out. Without the clip the same 100 passes keep raising every style that drew a click and
lowering every style that did not, until some ratios pass 3 and the 90 old samples are being
used as if the new policy had collected them. A gradient computed once and reused for every
pass fails the same way, because it never notices that a sample has dropped out.